# minigdb — Python API Walkthrough

This notebook covers the full Python API for minigdb:
- Local embedded database (`minigdb.open`)
- GQL queries: INSERT, MATCH, WHERE, ORDER BY, aggregation, path traversal
- Explicit transactions (BEGIN / COMMIT / ROLLBACK)
- DataFrame output (polars / pandas)
- Graph algorithms via `CALL ... YIELD`
- Remote server client (`minigdb.connect`)

---

## 1. Build and Install

The Python module is built from Rust using [maturin](https://github.com/PyO3/maturin).  
Run this once from the project root (not inside the notebook) before importing:

```bash
pip install maturin
maturin develop --features python
```

Optional DataFrame support:
```bash
pip install polars pandas
```

In [1]:
import tempfile, os
import minigdb
print("minigdb imported successfully")
print("Available:", [x for x in dir(minigdb) if not x.startswith('_')])

minigdb imported successfully
Available: ['MiniGdb', 'MiniGdbClient', 'connect', 'minigdb', 'open', 'open_at']


---
## 2. Opening a Database

Two ways to open a graph:

| Function | Description |
|---|---|
| `minigdb.open(name)` | Named graph in platform data dir (`~/.local/share/minigdb/graphs/<name>/`) |
| `minigdb.open_at(path)` | Explicit filesystem path — useful for notebooks/testing |

Both return a `MiniGdb` handle that supports context manager (`with`) syntax.

In [2]:
# Use a temp directory so the notebook is self-contained and repeatable.
# In real use: db = minigdb.open("my_graph")
tmp_dir = tempfile.mkdtemp(prefix="minigdb_example_")
print(f"Graph stored at: {tmp_dir}")

db = minigdb.open_at(tmp_dir)
print("Database opened:", db)

Graph stored at: /var/folders/w5/z2pj089s0s71rnvymj962ly00000gn/T/minigdb_example_6au0xraq
Database opened: <builtins.MiniGdb object at 0x107962c10>


In [3]:
# db = minigdb.open("test")

---
## 3. Inserting Nodes

Syntax: `INSERT (:Label {key: value, ...})`

Each node automatically gets a ULID (universally unique, time-ordered ID).
Supported value types: strings, integers, floats, booleans, null.

In [4]:
db.query('MATCH (a) DETACH DELETE a')

[{'result': 'Deleted 0 element(s) [txn 0]'}]

In [5]:
# Insert person nodes
db.query('INSERT (:Person {name: "Alice",  age: 31, active: true})')
db.query('INSERT (:Person {name: "Bob",    age: 25, active: true})')
db.query('INSERT (:Person {name: "Carol",  age: 35, active: false})')
db.query('INSERT (:Person {name: "Dave",   age: 22, active: true})')
db.query('INSERT (:Person {name: "Eve",    age: 29, active: true})')

# Insert company nodes
db.query('INSERT (:Company {name: "Acme Corp",   industry: "Technology"})')
db.query('INSERT (:Company {name: "Globex Inc",  industry: "Finance"})')

print("Nodes inserted.")

Nodes inserted.


In [6]:
# OPTIONAL MATCH — null if no match (like a LEFT JOIN)
# MATCH followed by OPTIONAL MATCH is supported as a chained clause.
rows = db.query(
    'MATCH (p:Person) '
    'OPTIONAL MATCH (p)-[:WORKS_AT]->(c:Company) '
    'RETURN p.name, c.name ORDER BY p.name'
)
print("People with optional company (None = not employed in graph):")
for r in rows:
    company = r.get('c.name') or '—'
    print(f"  {r['p.name']:10}  {company}")

People with optional company (None = not employed in graph):
  Alice       —
  Bob         —
  Carol       —
  Dave        —
  Eve         —


In [7]:
# Social connections between people
db.query('MATCH (a:Person {name:"Alice"}), (b:Person {name:"Bob"})   INSERT (a)-[:KNOWS {since: 2020}]->(b)')
db.query('MATCH (a:Person {name:"Alice"}), (c:Person {name:"Carol"}) INSERT (a)-[:KNOWS {since: 2018}]->(c)')
db.query('MATCH (b:Person {name:"Bob"}),   (d:Person {name:"Dave"})  INSERT (b)-[:KNOWS {since: 2022}]->(d)')
db.query('MATCH (c:Person {name:"Carol"}), (e:Person {name:"Eve"})   INSERT (c)-[:KNOWS {since: 2019}]->(e)')

# Employment relationships
db.query('MATCH (a:Person {name:"Alice"}), (co:Company {name:"Acme Corp"})  INSERT (a)-[:WORKS_AT {role: "Engineer",  salary: 95000}]->(co)')
db.query('MATCH (b:Person {name:"Bob"}),   (co:Company {name:"Acme Corp"})  INSERT (b)-[:WORKS_AT {role: "Analyst",   salary: 72000}]->(co)')
db.query('MATCH (c:Person {name:"Carol"}), (co:Company {name:"Globex Inc"}) INSERT (c)-[:WORKS_AT {role: "Manager",   salary: 110000}]->(co)')
db.query('MATCH (e:Person {name:"Eve"}),   (co:Company {name:"Globex Inc"}) INSERT (e)-[:WORKS_AT {role: "Developer", salary: 88000}]->(co)')

# Management
db.query('MATCH (a:Person {name:"Alice"}), (b:Person {name:"Bob"})  INSERT (a)-[:MANAGES]->(b)')
db.query('MATCH (c:Person {name:"Carol"}), (e:Person {name:"Eve"})  INSERT (c)-[:MANAGES]->(e)')

print("Edges inserted.")

Edges inserted.


---
## 5. Basic Queries

`query()` always returns a list of dicts — one dict per result row.

In [8]:
# All people
rows = db.query('MATCH (n:Person) RETURN n.name, n.age, n.active ORDER BY n.name')
print("All people:")
for r in rows:
    print(f"  {r['n.name']:10}  age={r['n.age']}  active={r['n.active']}")

All people:
  Alice       age=31  active=True
  Bob         age=25  active=True
  Carol       age=35  active=False
  Dave        age=22  active=True
  Eve         age=29  active=True


In [9]:
# All companies
rows = db.query('MATCH (c:Company) RETURN c.name, c.industry ORDER BY c.name')
print("Companies:")
for r in rows:
    print(f"  {r['c.name']:15}  {r['c.industry']}")

Companies:
  Acme Corp        Technology
  Globex Inc       Finance


---
## 14. Graph Algorithms

Algorithms are invoked via `CALL algorithmName(param: value) YIELD col1, col2`

**Important**: `CALL ... YIELD` is a complete statement — it cannot be followed by `RETURN`, `WHERE`, or `ORDER BY`. Post-process results in Python.  
All arguments are **named** (`source: "id"`, not positional).

| Algorithm | Example |
|---|---|
| BFS | `CALL bfs(source: "id") YIELD node, depth` |
| Shortest path | `CALL shortestPath(source: "id1", target: "id2") YIELD path, cost` |
| Weakly connected components | `CALL wcc() YIELD node, component` |
| PageRank | `CALL pageRank(iterations: 20, dampingFactor: 0.85) YIELD node, score` |
| Degree centrality | `CALL degreeCentrality() YIELD node, degree, in_degree, out_degree` |
| Betweenness | `CALL betweennessCentrality() YIELD node, score` |
| Triangle count | `CALL triangleCount() YIELD node, triangles` |
| Label propagation | `CALL labelPropagation() YIELD node, community` |

In [10]:
# PageRank — find the most influential people
# CALL...YIELD is a complete statement; filter by label in Python.
pr_rows = db.query('CALL pageRank(iterations: 20, dampingFactor: 0.85) YIELD node, score')
pr_scores = {r['node']: r['score'] for r in pr_rows}

# Resolve Person names and join with scores
person_rows = db.query('MATCH (n:Person) RETURN n, n.name')
top5 = sorted(
    [{'name': r['n.name'], 'pagerank': round(pr_scores.get(str(r['n']), 0) * 1000) / 1000}
     for r in person_rows if str(r['n']) in pr_scores],
    key=lambda r: r['pagerank'], reverse=True
)[:5]
print("PageRank (top people):")
for r in top5:
    print(f"  {r['name']:10}  pagerank={r['pagerank']}")

PageRank (top people):
  Eve         pagerank=0.146
  Dave        pagerank=0.139
  Bob         pagerank=0.124
  Carol       pagerank=0.105
  Alice       pagerank=0.087


In [11]:
# Degree centrality — who has the most connections?
# CALL...YIELD is a complete statement; sort in Python.
rows = db.query('CALL degreeCentrality() YIELD node, degree, in_degree, out_degree')
top5 = sorted(rows, key=lambda r: r['degree'], reverse=True)[:5]
print("Degree centrality (top 5 nodes):")
for r in top5:
    print(f"  node={str(r['node'])[:20]}  degree={r['degree']}  in={r['in_degree']}  out={r['out_degree']}")

Degree centrality (top 5 nodes):
  node=01KME1WDMPSESFYR1PHF  degree=4  in=0  out=4
  node=01KME1WDMQ3PBV8AZ5YZ  degree=4  in=2  out=2
  node=01KME1WDMQBTQ93FCNR2  degree=4  in=1  out=3
  node=01KME1WDMQNEZRTX204Y  degree=3  in=2  out=1
  node=01KME1WDMQAJK3JZEHVA  degree=2  in=2  out=0


In [12]:
# Weakly Connected Components — aggregate component sizes in Python.
from collections import Counter
rows = db.query('CALL wcc() YIELD node, component')
comp_counts = Counter(r['component'] for r in rows)
print("Connected components:")
for comp, size in sorted(comp_counts.items(), key=lambda x: -x[1]):
    print(f"  component {comp}: {size} node(s)")

Connected components:
  component 01KME1WDMPSESFYR1PHFCBZYE8: 7 node(s)


In [13]:
# Shortest path — all CALL args must be named (source:, target:).
alice_row = db.query('MATCH (n:Person {name:"Alice"}) RETURN n')
eve_row   = db.query('MATCH (n:Person {name:"Eve"})   RETURN n')

if alice_row and eve_row:
    alice_id = alice_row[0]['n']
    eve_id   = eve_row[0]['n']
    rows = db.query(f'CALL shortestPath(source: "{alice_id}", target: "{eve_id}") YIELD path, cost')
    print("Shortest path Alice→Eve:")
    for r in rows:
        print(f"  cost={r['cost']}  hops={len(r['path']) if isinstance(r['path'], list) else r['path']}")
else:
    print("Could not find Alice or Eve nodes")


Shortest path Alice→Eve:
  cost=2.0  hops=3


In [14]:
# Direct KNOWS relationships
rows = db.query(
    'MATCH (a:Person)-[r:KNOWS]->(b:Person) '
    'RETURN a.name, b.name, r.since ORDER BY a.name'
)
print("KNOWS edges:")
for r in rows:
    print(f"  {r['a.name']} → {r['b.name']}  (since {r['r.since']})") 

KNOWS edges:
  Alice → Carol  (since 2018)
  Alice → Bob  (since 2020)
  Bob → Dave  (since 2022)
  Carol → Eve  (since 2019)


In [15]:
# Who works where, with role and salary
rows = db.query(
    'MATCH (p:Person)-[w:WORKS_AT]->(c:Company) '
    'RETURN p.name, c.name, w.role, w.salary ORDER BY w.salary DESC'
)
print("Employment (highest salary first):")
for r in rows:
    print(f"  {r['p.name']:8} @ {r['c.name']:15}  {r['w.role']:12}  ${r['w.salary']:,}")

Employment (highest salary first):
  Carol    @ Globex Inc       Manager       $110,000
  Alice    @ Acme Corp        Engineer      $95,000
  Eve      @ Globex Inc       Developer     $88,000
  Bob      @ Acme Corp        Analyst       $72,000


In [16]:
# Any direction — undirected match
rows = db.query(
    'MATCH (a:Person)-[:MANAGES]-(b:Person) '
    'RETURN a.name, b.name'
)
print("MANAGES relationships (any direction):")
for r in rows:
    print(f"  {r['a.name']} ↔ {r['b.name']}")

MANAGES relationships (any direction):
  Alice ↔ Bob
  Bob ↔ Alice
  Carol ↔ Eve
  Eve ↔ Carol


# close() releases the RocksDB lock, so the same path can be reopened.
db.close()
print("Database closed.")

# Context manager — automatically calls close() on exit (checkpoints + releases lock).
with minigdb.open_at(tmp_dir) as db2:
    rows = db2.query('MATCH (n:Person) RETURN count(*) AS total')
    print(f"Re-opened graph — person count: {rows[0]['total']}")
# db2.close() called automatically here

In [17]:
# Who can Alice reach via KNOWS (any number of hops)?
rows = db.query(
    'MATCH (a:Person {name:"Alice"})-[:KNOWS*]->(b:Person) '
    'RETURN a.name, b.name'
)
print("Reachable from Alice via KNOWS:")
for r in rows:
    print(f"  {r['a.name']} → {r['b.name']}")

Reachable from Alice via KNOWS:
  Alice → Alice
  Alice → Carol
  Alice → Bob
  Alice → Eve
  Alice → Dave


In [18]:
# 2-hop KNOWS connections only
rows = db.query(
    'MATCH (a:Person {name:"Alice"})-[:KNOWS*2..2]->(b:Person) '
    'RETURN a.name, b.name'
)
print("Exactly 2 hops from Alice via KNOWS:")
for r in rows:
    print(f"  {r['a.name']} → (2 hops) → {r['b.name']}")

Exactly 2 hops from Alice via KNOWS:
  Alice → (2 hops) → Eve
  Alice → (2 hops) → Dave


---
## 9. Aggregation

Aggregate functions: `count(*)`, `count(expr)`, `sum`, `avg`, `min`, `max`, `collect`

Non-aggregate RETURN items form the implicit GROUP BY key.

In [19]:
# Overall stats
rows = db.query('MATCH (n:Person) RETURN count(*) AS total, avg(n.age) AS avg_age, min(n.age) AS youngest, max(n.age) AS oldest')
r = rows[0]
print(f"People: {r['total']}  avg age: {r['avg_age']:.1f}  youngest: {r['youngest']}  oldest: {r['oldest']}")

People: 5  avg age: 28.4  youngest: 22  oldest: 35


In [20]:
# Average salary per company (GROUP BY c.name implicitly)
rows = db.query(
    'MATCH (p:Person)-[w:WORKS_AT]->(c:Company) '
    'RETURN c.name, count(*) AS headcount, avg(w.salary) AS avg_salary '
    'ORDER BY avg_salary DESC'
)
print("Avg salary by company:")
for r in rows:
    print(f"  {r['c.name']:15}  {r['headcount']} employees  avg ${r['avg_salary']:,.0f}")

Avg salary by company:
  Globex Inc       2 employees  avg $99,000
  Acme Corp        2 employees  avg $83,500


In [21]:
# collect() — gather values into a list
rows = db.query(
    'MATCH (p:Person)-[:WORKS_AT]->(c:Company) '
    'RETURN c.name, collect(p.name) AS employees ORDER BY c.name'
)
print("Employees per company:")
for r in rows:
    print(f"  {r['c.name']}: {sorted(r['employees'])}")

Employees per company:
  Acme Corp: ['Alice', 'Bob']
  Globex Inc: ['Carol', 'Eve']


---
## 10. Updating and Deleting

In [22]:
# SET — update a property
db.query('MATCH (n:Person {name:"Bob"}) SET n.age = 26')
rows = db.query('MATCH (n:Person {name:"Bob"}) RETURN n.name, n.age')
print("Bob after SET:", rows[0])

Bob after SET: {'n.age': 26, 'n.name': 'Bob'}


In [23]:
# SET multiple properties at once
db.query('MATCH (n:Person {name:"Dave"}) SET n.age = 23, n.active = false')
rows = db.query('MATCH (n:Person {name:"Dave"}) RETURN n.name, n.age, n.active')
print("Dave after SET:", rows[0])

Dave after SET: {'n.name': 'Dave', 'n.age': 23, 'n.active': False}


In [24]:
# REMOVE — delete a single property
db.query('MATCH (n:Person {name:"Dave"}) REMOVE n.active')
rows = db.query('MATCH (n:Person {name:"Dave"}) RETURN n.name, n.age')
print("Dave after REMOVE active:", rows[0])

Dave after REMOVE active: {'n.name': 'Dave', 'n.age': 23}


In [25]:
# DELETE — remove a node (must have no edges, or use DETACH DELETE)
db.query('MATCH (n:Person {name:"Dave"}) DETACH DELETE n')
rows = db.query('MATCH (n:Person) RETURN n.name ORDER BY n.name')
print("People after deleting Dave:", [r['n.name'] for r in rows])

People after deleting Dave: ['Alice', 'Bob', 'Carol', 'Eve']


---
## 11. Explicit Transactions

By default each `query()` call auto-commits.  
Use `begin()` / `commit()` / `rollback()` for atomic multi-statement batches.

In [26]:
# Atomic batch — all or nothing
db.begin()
db.query('INSERT (:Person {name: "Frank", age: 40, active: true})')
db.query('INSERT (:Person {name: "Grace", age: 34, active: true})')
db.query('MATCH (f:Person {name:"Frank"}), (g:Person {name:"Grace"}) INSERT (f)-[:KNOWS {since: 2021}]->(g)')
db.commit()

rows = db.query('MATCH (n:Person) RETURN n.name ORDER BY n.name')
print("People after committed batch:", [r['n.name'] for r in rows])

People after committed batch: ['Alice', 'Bob', 'Carol', 'Eve', 'Frank', 'Grace']


In [27]:
# Rollback — nothing gets written
db.begin()
db.query('INSERT (:Person {name: "Ghost", age: 0})')
db.rollback()

rows = db.query('MATCH (n:Person {name:"Ghost"}) RETURN n.name')
print("Ghost after rollback (should be empty):", rows)

Ghost after rollback (should be empty): []


---
## 12. Advanced GQL — UNWIND, WITH, OPTIONAL MATCH

In [28]:
# UNWIND — expand a list into rows
rows = db.query('UNWIND [10, 20, 30] AS x RETURN x, x * x AS square')
print("UNWIND example:")
for r in rows:
    print(f"  x={r['x']}  x²={r['square']}")

UNWIND example:
  x=10  x²=100
  x=20  x²=400
  x=30  x²=900


In [29]:
# OPTIONAL MATCH — null if no match (like a LEFT JOIN)
rows = db.query(
    'MATCH (p:Person) '
    'OPTIONAL MATCH (p)-[:WORKS_AT]->(c:Company) '
    'RETURN p.name, c.name ORDER BY p.name'
)
print("People with optional company (None = not employed in graph):")
for r in rows:
    company = r.get('c.name') or '—'
    print(f"  {r['p.name']:10}  {company}")

People with optional company (None = not employed in graph):
  Alice       Acme Corp
  Bob         Acme Corp
  Carol       Globex Inc
  Eve         Globex Inc
  Frank       —
  Grace       —


In [30]:
# MATCH ... WITH ... MATCH — pipeline / subquery pattern
rows = db.query(
    'MATCH (p:Person)-[:WORKS_AT]->(c:Company) '
    'WITH c.name AS company, count(p) AS headcount '
    'WHERE headcount > 1 '
    'RETURN company, headcount ORDER BY headcount DESC'
)
print("Companies with more than 1 employee:")
for r in rows:
    print(f"  {r['company']}: {r['headcount']} people")

Companies with more than 1 employee:
  Acme Corp: 2 people
  Globex Inc: 2 people


---
## 13. DataFrame Output

Use `query_df()` (polars) or `query_pandas()` (pandas) for tabular analysis.

In [31]:
# polars DataFrame
try:
    df = db.query_df(
        'MATCH (p:Person)-[w:WORKS_AT]->(c:Company) '
        'RETURN p.name, p.age, c.name, w.role, w.salary '
        'ORDER BY w.salary DESC'
    )
    print("polars DataFrame:")
    print(df)
except RuntimeError as e:
    print(f"polars not installed: {e}")
    print("Run: pip install polars")

polars DataFrame:
shape: (4, 5)
┌───────┬───────────┬────────┬──────────┬────────────┐
│ p.age ┆ w.role    ┆ p.name ┆ w.salary ┆ c.name     │
│ ---   ┆ ---       ┆ ---    ┆ ---      ┆ ---        │
│ i64   ┆ str       ┆ str    ┆ i64      ┆ str        │
╞═══════╪═══════════╪════════╪══════════╪════════════╡
│ 35    ┆ Manager   ┆ Carol  ┆ 110000   ┆ Globex Inc │
│ 31    ┆ Engineer  ┆ Alice  ┆ 95000    ┆ Acme Corp  │
│ 29    ┆ Developer ┆ Eve    ┆ 88000    ┆ Globex Inc │
│ 26    ┆ Analyst   ┆ Bob    ┆ 72000    ┆ Acme Corp  │
└───────┴───────────┴────────┴──────────┴────────────┘


In [32]:
# pandas DataFrame
try:
    df = db.query_pandas(
        'MATCH (p:Person) RETURN p.name, p.age, p.active ORDER BY p.age'
    )
    print("pandas DataFrame:")
    print(df.to_string(index=False))
except RuntimeError as e:
    print(f"pandas not installed: {e}")
    print("Run: pip install pandas")

pandas DataFrame:
p.name  p.age  p.active
   Bob     26      True
   Eve     29      True
 Alice     31      True
 Grace     34      True
 Carol     35     False
 Frank     40      True


---
## 14. Graph Algorithms

Algorithms are invoked via `CALL algorithmName(...) YIELD column1, column2`

| Algorithm | Call |
|---|---|
| BFS | `CALL bfs('startId') YIELD node, depth` |
| Shortest path | `CALL shortestPath('src', 'tgt') YIELD path, cost` |
| Weakly connected components | `CALL wcc() YIELD node, component` |
| PageRank | `CALL pageRank() YIELD node, score` |
| Degree centrality | `CALL degreeCentrality() YIELD node, degree, in_degree, out_degree` |
| Betweenness | `CALL betweennessCentrality() YIELD node, score` |
| Triangle count | `CALL triangleCount() YIELD node, triangles` |
| Label propagation | `CALL labelPropagation() YIELD node, community` |

In [33]:
# PageRank — find the most influential people
rows = db.query(
    'CALL pageRank(iterations: 20, dampingFactor: 0.85) YIELD node, score '
    'MATCH (n) WHERE n = node AND "Person" IN labels(n) '
    'RETURN n.name, round(score * 1000) / 1000 AS pagerank '
    'ORDER BY pagerank DESC LIMIT 5'
)
if rows:
    print("PageRank (top people):")
    for r in rows:
        print(f"  {r['n.name']:10}  {r['pagerank']}")
else:
    # Simple CALL without label filter
    rows = db.query('CALL pageRank() YIELD node, score RETURN node, score ORDER BY score DESC LIMIT 5')
    print("PageRank scores:", rows)

PageRank (top people):
  Alice       0.075
  Bob         0.107
  Carol       0.091
  Eve         0.126
  Grace       0.138


In [34]:
# Degree centrality — who has the most connections?
rows = db.query(
    'CALL degreeCentrality() YIELD node, degree, in_degree, out_degree '
    'RETURN node, degree, in_degree, out_degree '
    'ORDER BY degree DESC LIMIT 5'
)
print("Degree centrality (top 5 nodes):")
for r in rows:
    print(f"  node={str(r['node'])[:20]}  degree={r['degree']}  in={r['in_degree']}  out={r['out_degree']}")

Degree centrality (top 5 nodes):
  node=01KME1WDMPSESFYR1PHF  degree=4  in=0  out=4
  node=01KME1WDMQBTQ93FCNR2  degree=4  in=1  out=3
  node=01KME1WDMQ3PBV8AZ5YZ  degree=3  in=2  out=1
  node=01KME1WDMQNEZRTX204Y  degree=3  in=2  out=1
  node=01KME1WDMQAJK3JZEHVA  degree=2  in=2  out=0


In [35]:
# Weakly Connected Components — which nodes form clusters?
rows = db.query('CALL wcc() YIELD node, component RETURN component, count(*) AS size ORDER BY size DESC')
print("Connected components:")
for r in rows:
    print(f"  component {r['component']}: {r['size']} node(s)")

Connected components:
  component 01KME1WDMPSESFYR1PHFCBZYE8: 6 node(s)
  component 01KME1WDQFGPEE9DBX69ZM42X2: 2 node(s)


In [36]:
# Shortest path — all CALL args must be named (source:, target:).
alice_row = db.query('MATCH (n:Person {name:"Alice"}) RETURN n')
eve_row   = db.query('MATCH (n:Person {name:"Eve"})   RETURN n')

if alice_row and eve_row:
    alice_id = alice_row[0]['n']
    eve_id   = eve_row[0]['n']
    rows = db.query(f'CALL shortestPath(source: "{alice_id}", target: "{eve_id}") YIELD path, cost')
    print("Shortest path Alice→Eve:")
    for r in rows:
        print(f"  cost={r['cost']}  hops={len(r['path']) if isinstance(r['path'], list) else r['path']}")
else:
    print("Could not find Alice or Eve nodes")


Shortest path Alice→Eve:
  cost=2.0  hops=3


---
## 15. Property Index

Indexes speed up `MATCH (n:Label {prop: value})` lookups from O(n) to O(1).

In [37]:
# Create an index on Person.name
db.query('CREATE INDEX ON :Person(name)')

# List all indexes
rows = db.query('SHOW INDEXES')
print("Indexes:")
for r in rows:
    print(f"  {r}")

# Query now uses the index automatically
rows = db.query('MATCH (n:Person {name:"Alice"}) RETURN n.name, n.age')
print("\nIndexed lookup result:", rows)

Indexes:
  {'label': 'Person', 'property': 'name', 'entries': 6}

Indexed lookup result: [{'n.name': 'Alice', 'n.age': 31}]


---
## 16. Context Manager and Cleanup

In [38]:
# Close the manual handle from earlier
db.close()
print("Database closed.")

# Context manager — automatically calls close() on exit
with minigdb.open_at(tmp_dir) as db2:
    rows = db2.query('MATCH (n:Person) RETURN count(*) AS total')
    print(f"Re-opened graph — person count: {rows[0]['total']}")
# db2 is closed and checkpointed here automatically

Database closed.
Re-opened graph — person count: 6


---
## 17. Remote Server Client

If the minigdb server is running (`minigdb serve`), connect via TCP:

```bash
# Start the server first (in a separate terminal):
minigdb serve --port 7474 --no-auth
```

`MiniGdbClient` has the same `query()`, `query_df()`, `query_pandas()`, `begin()`, `commit()`, `rollback()` interface as the local `MiniGdb` — swap transparently.

In [39]:
# Remote connection (only runs if server is up)
# The server is a separate database, so we insert a small dataset first,
# then query it — demonstrating that the client API is identical to the local API.
SERVER = "localhost:7474"

try:
    with minigdb.connect(SERVER) as client:
        # Seed the server graph with the same people used in this notebook.
        client.query('MATCH (n) DETACH DELETE n')
        client.query('INSERT (:Person {name: "Alice", age: 31})')
        client.query('INSERT (:Person {name: "Bob",   age: 25})')
        client.query('INSERT (:Person {name: "Carol", age: 35})')
        client.query('INSERT (:Person {name: "Dave",  age: 22})')
        client.query('INSERT (:Person {name: "Eve",   age: 29})')
        rows = client.query('MATCH (n:Person) RETURN n.name, n.age ORDER BY n.name')
        print("Remote query result (5 people inserted via client):")
        for r in rows:
            print(f"  name = {r['n.name']:10}  age = {r['n.age']}")
except OSError as e:
    print(f"Server not running at {SERVER}: {e}")
    print("Start with: minigdb serve --port 7474 --no-auth")


Remote query result (5 people inserted via client):
  name = Alice       age = 31
  name = Bob         age = 25
  name = Carol       age = 35
  name = Dave        age = 22
  name = Eve         age = 29


---
## 18. Cleanup

In [40]:
import shutil
shutil.rmtree(tmp_dir)
print(f"Cleaned up temporary database at {tmp_dir}")

Cleaned up temporary database at /var/folders/w5/z2pj089s0s71rnvymj962ly00000gn/T/minigdb_example_6au0xraq
